# MLP 2x100 ReLU

Dos capas ocultas de 100 neuronas para comprobar si mas profundidad mejora frente a una sola capa.

Entrena esta arquitectura para las 16 combinaciones de ventanas y guarda resultados en `data/mlp/mlp_2x100_relu.csv`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "util.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "util.py").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error

from util import (
    get_train_test,
    RANDOM_SEED,
    compare_to_benchmark,
    plot_benchmark_comparison,
    configure_mlflow,
    log_keras_grid_run,
)

keras.utils.set_random_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [2]:
MODEL_NAME = "mlp_2x100_relu"
EXPERIMENT_NAME = "MLP"
LOG_MODEL_ARTIFACT = False
ARCHITECTURE_PARAMS = {
    'hidden_layers': 2,
    'neurons': 100,
    'activation': 'relu',
    'regularization': 'none',
}
mlflow = configure_mlflow(EXPERIMENT_NAME)
RESULTS_PATH = PROJECT_ROOT / "data" / "mlp" / f"{MODEL_NAME}.csv"
HISTORY_PATH = PROJECT_ROOT / "data" / "mlp" / f"{MODEL_NAME}_history.csv"

input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]

EPOCHS = 500
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
VALIDATION_SPLIT = 0.1

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS_PATH)
print("history:", HISTORY_PATH)


results: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/mlp/mlp_2x100_relu.csv
history: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/mlp/mlp_2x100_relu_history.csv


## Definicion de la red

In [3]:
def build_model(input_dim, output_dim):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(100, activation="relu"))
    model.add(Dense(100, activation="relu"))
    model.add(Dense(output_dim))
    model.compile(loss="mean_absolute_error", optimizer=Adam(learning_rate=LEARNING_RATE))
    return model


## Entrenamiento en grid de ventanas

In [4]:
results = []
history_rows = []

for in_w in input_windows:
    for out_w in output_windows:
        d = get_train_test(input_window_size=in_w, output_window_size=out_w)

        X_train = d.X_train.reshape(d.X_train.shape[0], -1)
        X_test = d.X_test.reshape(d.X_test.shape[0], -1)

        keras.utils.set_random_seed(RANDOM_SEED)
        model = build_model(input_dim=X_train.shape[1], output_dim=d.y_train.shape[1])


        hist = model.fit(
            X_train,
            d.y_train,
            validation_split=VALIDATION_SPLIT,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0,
            shuffle=False,
        )

        y_pred_train = model.predict(X_train, verbose=0)
        y_pred_test = model.predict(X_test, verbose=0)

        row = {
            "model_name": MODEL_NAME,
            "input_window": in_w,
            "output_window": out_w,
            "MAE_train": mean_absolute_error(d.y_train, y_pred_train),
            "MAE_val": min(hist.history["val_loss"]),
            "MAE_test": mean_absolute_error(d.y_test, y_pred_test),
            "epochs": len(hist.history["loss"]),
            "n_params": model.count_params(),
        }
        results.append(row)

        for epoch, (loss, val_loss) in enumerate(
            zip(hist.history["loss"], hist.history["val_loss"]), start=1
        ):
            history_rows.append({
                "model_name": MODEL_NAME,
                "input_window": in_w,
                "output_window": out_w,
                "epoch": epoch,
                "loss": loss,
                "val_loss": val_loss,
            })

        run_name = f"{MODEL_NAME}_input{in_w}_output{out_w}"
        log_keras_grid_run(
            mlflow=mlflow,
            model=model,
            history=hist,
            run_name=run_name,
            model_name=MODEL_NAME,
            input_window=in_w,
            output_window=out_w,
            metrics_row=row,
            train_shape=X_train.shape,
            test_shape=X_test.shape,
            output_dim=d.y_train.shape[1],
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            validation_split=VALIDATION_SPLIT,
            extra_params=ARCHITECTURE_PARAMS,
            log_model_artifact=LOG_MODEL_ARTIFACT,
        )

        results_df = pd.DataFrame(results)
        history_df = pd.DataFrame(history_rows)
        results_df.to_csv(RESULTS_PATH, index=False)
        history_df.to_csv(HISTORY_PATH, index=False)

        print(
            f"input={in_w:>3} output={out_w:>3} | "
            f"train={row['MAE_train']:.6f} val={row['MAE_val']:.6f} "
            f"test={row['MAE_test']:.6f} epochs={row['epochs']} "
            f"params={row['n_params']}"
        )

results_df = pd.DataFrame(results)
history_df = pd.DataFrame(history_rows)
results_df


input=  5 output=  1 | train=0.009897 val=0.009036 test=0.015833 epochs=500 params=24023


input=  5 output=  5 | train=0.004478 val=0.004167 test=0.006956 epochs=500 params=24023


input=  5 output= 30 | train=0.001934 val=0.001764 test=0.002895 epochs=500 params=24023


input=  5 output= 90 | train=0.001305 val=0.001025 test=0.001692 epochs=500 params=24023


input= 10 output=  1 | train=0.009137 val=0.009030 test=0.015257 epochs=500 params=35523


...

input= 90 output=  1 | train=0.007493 val=0.009032 test=0.015201 epochs=500 params=219523


input= 90 output=  5 | train=0.003785 val=0.004169 test=0.006474 epochs=500 params=219523


input= 90 output= 30 | train=0.001931 val=0.001773 test=0.002668 epochs=500 params=219523


input= 90 output= 90 | train=0.001278 val=0.001033 test=0.001425 epochs=500 params=219523


,model_name,input_window,output_window,MAE_train,MAE_val,MAE_test,epochs,n_params
0,mlp_2x100_relu,5,1,0.009897,0.009036,0.015833,500,24023
1,mlp_2x100_relu,5,5,0.004478,0.004167,0.006956,500,24023
2,mlp_2x100_relu,5,30,0.001934,0.001764,0.002895,500,24023
3,mlp_2x100_relu,5,90,0.001305,0.001025,0.001692,500,24023
4,mlp_2x100_relu,10,1,0.009137,0.009030,0.015257,500,35523
5,mlp_2x100_relu,10,5,0.004303,0.004167,0.006807,500,35523
6,mlp_2x100_relu,10,30,0.001972,0.001764,0.002937,500,35523
7,mlp_2x100_relu,10,90,0.001238,0.001029,0.001649,500,35523
8,mlp_2x100_relu,30,1,0.008153,0.009025,0.015640,500,81523
9,mlp_2x100_relu,30,5,0.004077,0.004161,0.006893,500,81523


## Matrices de error

In [5]:
mae_train_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_train")
mae_val_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_val")
mae_test_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_test")

print("MAE train")
display(mae_train_matrix)
print("MAE val")
display(mae_val_matrix)
print("MAE test")
display(mae_test_matrix)


MAE train


output_window,1,5,30,90
input_window,,,,
5,0.009897,0.004478,0.001934,0.001305
10,0.009137,0.004303,0.001972,0.001238
30,0.008153,0.004077,0.002020,0.001452
90,0.007493,0.003785,0.001931,0.001278


MAE val


output_window,1,5,30,90
input_window,,,,
5,0.009036,0.004167,0.001764,0.001025
10,0.009030,0.004167,0.001764,0.001029
30,0.009025,0.004161,0.001762,0.001031
90,0.009032,0.004169,0.001773,0.001033


MAE test


output_window,1,5,30,90
input_window,,,,
5,0.015833,0.006956,0.002895,0.001692
10,0.015257,0.006807,0.002937,0.001649
30,0.015640,0.006893,0.002929,0.001587
90,0.015201,0.006474,0.002668,0.001425


## Heatmaps

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
matrices = [
    (mae_train_matrix, "MAE train"),
    (mae_val_matrix, "MAE val"),
    (mae_test_matrix, "MAE test"),
]
vmin = min(matrix.values.min() for matrix, _ in matrices)
vmax = max(matrix.values.max() for matrix, _ in matrices)

for ax, (matrix, title) in zip(axes, matrices):
    im = ax.imshow(matrix.values, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns)
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.set_xlabel("Ventana salida")
    ax.set_ylabel("Ventana entrada")
    ax.set_title(f"{title} - {MODEL_NAME}")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix.values[i, j]:.4f}", ha="center", va="center", color="white", fontsize=9)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()


/var/folders/py/c5_xfbqn469g5_844mv32gt40000gn/T/ipykernel_59574/2336802165.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Curvas de aprendizaje

In [7]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14), sharey=False)
axes = axes.ravel()

for ax, ((in_w, out_w), group) in zip(
    axes,
    history_df.groupby(["input_window", "output_window"], sort=True),
):
    ax.plot(group["epoch"], group["loss"], label="train")
    ax.plot(group["epoch"], group["val_loss"], label="val")
    ax.set_title(f"in={in_w}, out={out_w}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MAE")
    ax.grid(True, alpha=0.3)

axes[0].legend()
plt.suptitle(f"Curvas de aprendizaje - {MODEL_NAME}", y=1.02)
plt.tight_layout()
plt.show()


/var/folders/py/c5_xfbqn469g5_844mv32gt40000gn/T/ipykernel_59574/1319896804.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Comparacion contra regresion lineal

In [8]:
comparison = compare_to_benchmark(results_df, benchmark="lr_benchmark")
display(comparison)

plot_benchmark_comparison(results_df, benchmark="lr_benchmark", model_name=MODEL_NAME)
plt.show()


,input_window,output_window,MAE_test,MAE_test_benchmark,delta,pct_delta
0,5,1,0.015833,0.012384,0.003449,27.853043
1,5,5,0.006956,0.005625,0.001331,23.661581
2,5,30,0.002895,0.002340,0.000555,23.703125
3,5,90,0.001692,0.001271,0.000420,33.057714
4,10,1,0.015257,0.012554,0.002703,21.530822
5,10,5,0.006807,0.005698,0.001109,19.468015
6,10,30,0.002937,0.002358,0.000578,24.526726
7,10,90,0.001649,0.001282,0.000367,28.604438
8,30,1,0.015640,0.012924,0.002716,21.011178
9,30,5,0.006893,0.005877,0.001017,17.298297


/var/folders/py/c5_xfbqn469g5_844mv32gt40000gn/T/ipykernel_59574/4288889828.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
